In [1]:
#!/usr/bin/env python
"""
Analysis Pipeline: Produces Every Figure and Table in the Paper

This is the single script that turns embeddings (from 04_compute_embeddings.py)
into every statistical result, figure, and table reported in the paper.
Run it once, after embeddings exist for all four models.

What this script does, in order:
  1. Loads generation data + embeddings for all four models.
  2. Builds "paired items": for each segment/strategy/model/run, matches
     the event-condition generation to its no_event-condition counterpart,
     and computes the Multimodal Advantage Index (MAI) for each pair.
  3. Produces Figure 1: a boxplot of per-item MAI, pooled across
     prompting strategies, one box per model (median/IQR/full range,
     with a Shapiro-Wilk normality check to justify this choice over a
     mean-centered summary).
  4. Produces Figure 2: marginal MAI by model (bar chart, red = significant).
  5. Runs the global tutor-response-style clustering (fit once on all
     four models' pooled tutor utterances, propagated back to each model)
     and computes linguistic features to characterize each cluster.
  6. Produces Figure 3: a two-panel figure (MAI by strategy | MAI by
     tutor-response cluster), across models.
  7. Produces Table: aggregate similarity + MAI by model x strategy.
  8. Produces Table: MAI by tutor-response cluster, WITH Cohen's d
     (one-sample standardized effect size) alongside each p-value, so a
     reader can see practical magnitude, not just statistical significance.

Output directory: analysis_outputs/
"""

import os
import re

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway, shapiro
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# ══════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE CONFIG
#
# Set ONCE, here, and never overridden anywhere else in this script. This
# matters: if individual plotting functions set their own figsize/fontsize,
# figures end up visually inconsistent with each other (this happened
# during earlier drafts of this analysis and had to be fixed by
# consolidating everything into one script with one style block).
# ══════════════════════════════════════════════════════════════════════════
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300,
})

# ══════════════════════════════════════════════════════════════════════════
# CONFIG -- edit these paths for your environment
# ══════════════════════════════════════════════════════════════════════════
FILE_TAGS = [
    "llm_mistrallatest", "llm_wizardlm27b", "llm_qwen3-vl8b", "llm_gemma327b"
]
DATA_DIR = "."  # directory containing llm_<tag>.csv files
INPUT_FILES = {
    "llm_mistrallatest": f"{DATA_DIR}/llm_mistrallatest.csv",
    "llm_wizardlm27b":   f"{DATA_DIR}/llm_wizardlm27b.csv",
    "llm_qwen3-vl8b":    f"{DATA_DIR}/llm_qwen3-vl8b.csv",
    "llm_gemma327b":     f"{DATA_DIR}/llm_gemma327b.csv",
}
EMBED_DIR = "embeddings"        # output of 04_compute_embeddings.py
OUTPUT_DIR = "analysis_outputs"
EXPERIMENTS = ["baseline", "fewshot", "topic_context"]
K_RANGE = range(2, 20)           # candidate cluster counts to silhouette-search over
RANDOM_STATE = 42
SHAPIRO_SUBSAMPLE = 5000         # scipy's Shapiro test gets unreliable/slow above ~5000 points

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════
# SECTION 1: Load data + embeddings
#
# Each model has: a CSV of every generation (from 03_generate_llm_responses.py)
# and matching .npy embedding arrays (from 04_compute_embeddings.py), where
# array row i corresponds exactly to CSV row i. We reconstruct
# gen_tutor_sim (the core similarity score) here as a dot product, since
# embeddings were saved already L2-normalized -- normalized dot product IS
# cosine similarity, no separate normalization step needed at this stage.
# ══════════════════════════════════════════════════════════════════════════
def load_file(file_tag, input_csv):
    df = pd.read_csv(input_csv)
    # Resumed generation runs occasionally wrote a stray duplicate header
    # row mid-file -- filter those out before doing anything else.
    if "experiment" in df.columns:
        df = df[df["experiment"] != "experiment"].reset_index(drop=True)
    if "observation_id" in df.columns:
        df["observation_id"] = pd.to_numeric(df["observation_id"], errors="coerce")

    gen = np.load(os.path.join(EMBED_DIR, f"generation_emb__{file_tag}.npy"))
    tut = np.load(os.path.join(EMBED_DIR, f"tutor_emb__{file_tag}.npy"))
    ctx = np.load(os.path.join(EMBED_DIR, f"context_emb__{file_tag}.npy"))

    assert len(df) == len(gen) == len(tut) == len(ctx), \
        f"Row count mismatch for {file_tag}! CSV and embedding arrays must be the same length."

    # Since both `gen` and `tut` are unit-length (L2-normalized) vectors,
    # their row-wise dot product IS the cosine similarity between them.
    df["gen_tutor_sim"] = (gen * tut).sum(axis=1)
    df["file_tag"] = file_tag
    return df, gen, tut, ctx


all_dfs, all_gens, all_tuts, all_ctxs = {}, {}, {}, {}
for tag in FILE_TAGS:
    df, gen, tut, ctx = load_file(tag, INPUT_FILES[tag])
    all_dfs[tag], all_gens[tag], all_tuts[tag], all_ctxs[tag] = df, gen, tut, ctx
    print(f"{tag}: {len(df)} rows loaded")


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2: Build paired items and compute MAI
#
# For a given model + strategy, this joins each "event" condition
# generation to its matched "no_event" counterpart -- matched on the SAME
# transcript, segment, AND run index (not just any no_event row for that
# segment), so we're never accidentally comparing across different
# generation runs. MAI is then just the similarity difference within
# each matched pair.
# ══════════════════════════════════════════════════════════════════════════
def event_effect_paired_df(df, experiment, sim_col="gen_tutor_sim"):
    sub = df[df["experiment"] == experiment].copy()
    event_df = sub[sub["context_condition"] == "event"].copy()
    no_event_df = sub[sub["context_condition"] == "no_event"].copy()

    # Within each (transcriptID, segment_id) group, the two independent
    # generation runs get run_idx 0 and 1 based on the order they appear
    # in the CSV -- this recovers "which run" each row belongs to, since
    # that isn't stored as an explicit column.
    merge_cols = ["transcriptID", "segment_id"]
    event_df["run_idx"] = event_df.groupby(merge_cols).cumcount()
    no_event_df["run_idx"] = no_event_df.groupby(merge_cols).cumcount()
    merge_cols = merge_cols + ["run_idx"]

    paired = event_df.merge(
        no_event_df[merge_cols + [sim_col, "LLM_generation", "next_tutor_utterance"]],
        on=merge_cols, suffixes=("_event", "_no")
    )
    if paired.empty:
        return paired

    # This one line IS the Multimodal Advantage Index: positive means the
    # event condition was closer to the real tutor's utterance for this
    # specific matched pair; negative means no_event won for this pair.
    paired["diff"] = paired[f"{sim_col}_event"] - paired[f"{sim_col}_no"]
    return paired


# Build every model x strategy pairing once, reused by every section below
# rather than recomputed each time it's needed.
all_paired = {}
for tag, df in all_dfs.items():
    for exp in EXPERIMENTS:
        paired = event_effect_paired_df(df, exp)
        if not paired.empty:
            all_paired[(tag, exp)] = paired


# ══════════════════════════════════════════════════════════════════════════
# FIGURE 1: Per-item MAI distribution, one boxplot per model
#
# Design choice: pooled across all three prompting strategies into ONE box
# per model (not broken out by strategy within the figure) -- that
# breakdown is Figure 3 instead. Boxes are drawn HORIZONTALLY, with
# whis=(0,100) so the whiskers show the TRUE min/max, not the default
# 1.5*IQR rule -- because a chunk of this data is genuinely heavy-tailed,
# and hiding outliers past 1.5*IQR would misrepresent how wide the spread
# of dialogue-level heterogeneity actually is.
#
# Before trusting median/IQR as a summary, we run Shapiro-Wilk on each
# model's pooled distribution to check whether it's close enough to normal
# that a mean/SD summary would ALSO be defensible (median/IQR is safe
# either way, but this gives the reader an explicit justification).
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}\nFIGURE 1: Shapiro-Wilk normality check + per-model MAI distribution\n{'=' * 70}")

shapiro_rows = []
boxplot_data, boxplot_labels = [], []

for tag in FILE_TAGS:
    # Pool this model's MAI values across ALL THREE strategies into one
    # distribution -- this is what makes Figure 1 "one box per model"
    # rather than "one box per model x strategy".
    pooled = pd.concat([all_paired[(tag, exp)]["diff"] for exp in EXPERIMENTS if (tag, exp) in all_paired])
    pooled = pooled.dropna()

    # Shapiro-Wilk needs a reasonably-sized-but-not-huge sample to run
    # reliably and quickly; subsample rather than run on the full pool.
    sample = pooled.sample(min(SHAPIRO_SUBSAMPLE, len(pooled)), random_state=RANDOM_STATE)
    w_stat, w_p = shapiro(sample)

    shapiro_rows.append({
        "model": tag.replace("llm_", ""), "n": len(pooled),
        "median": pooled.median(), "q1": pooled.quantile(0.25), "q3": pooled.quantile(0.75),
        "min": pooled.min(), "max": pooled.max(),
        "shapiro_W": w_stat, "shapiro_p": w_p, "approx_normal": w_p > 0.05,
    })
    print(f"  {tag.replace('llm_', ''):15s} median={pooled.median():+.4f}  "
          f"IQR=[{pooled.quantile(0.25):+.4f}, {pooled.quantile(0.75):+.4f}]  "
          f"W={w_stat:.4f}  p={w_p:.4g}  {'(approx. normal)' if w_p > 0.05 else '(not normal)'}")

    boxplot_data.append(pooled.values)
    boxplot_labels.append(tag.replace("llm_", ""))

pd.DataFrame(shapiro_rows).to_csv(os.path.join(OUTPUT_DIR, "figure1_shapiro_results.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(
    boxplot_data, labels=boxplot_labels,
    whis=(0, 100),         # whiskers = true min/max, not the 1.5*IQR default
    vert=False,             # horizontal boxes: models stack top-to-bottom
    widths=0.6,             # wider boxes so the (often thin) IQR is still visible
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", edgecolor="black", linewidth=1.5, alpha=0.9),
    whiskerprops=dict(color="gray", linewidth=1, alpha=0.7),  # de-emphasize whiskers vs. the box
    capprops=dict(color="gray", linewidth=1, alpha=0.7),
    medianprops=dict(color="darkorange", linewidth=2.5),      # bright median line, easy to spot
)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("MAI (event $-$ no\\_event)")
ax.set_title("Per-Item MAI Distribution by Model\n(median, IQR, and full range, pooled across strategies)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure1_mai_distribution.png"), bbox_inches="tight")
plt.close()
print("\nSaved: figure1_mai_distribution.png")
print("Saved: figure1_shapiro_results.csv")


# ══════════════════════════════════════════════════════════════════════════
# FIGURE 2: Marginal MAI by model
#
# "Marginal" = collapsed across all three prompting strategies, testing
# whether MODEL IDENTITY ALONE (independent of strategy) predicts MAI.
# This is deliberately a SEPARATE figure from the strategy/cluster
# breakdown (Figure 3) -- combining all three into one crowded figure was
# tried earlier and made each panel too small to read clearly.
# ══════════════════════════════════════════════════════════════════════════
mai_by_model_rows = []
for tag in FILE_TAGS:
    all_diffs = [all_paired[(tag, exp)]["diff"] for exp in EXPERIMENTS if (tag, exp) in all_paired]
    diff = pd.concat(all_diffs, ignore_index=True)
    w_stat, w_p = stats.wilcoxon(diff)  # one-sample Wilcoxon: is the median MAI different from 0?
    mai_by_model_rows.append({
        "model": tag, "n": len(diff),
        "mean_MAI": diff.mean(), "std_MAI": diff.std(), "wilcoxon_p": w_p,
    })
mai_by_model_df = pd.DataFrame(mai_by_model_rows).sort_values("mean_MAI", ascending=False)

# One-way ANOVA: does mean MAI differ across the four models at all?
groups = [pd.concat([all_paired[(tag, exp)]["diff"] for exp in EXPERIMENTS if (tag, exp) in all_paired]).values
          for tag in FILE_TAGS]
f_stat, f_p = f_oneway(*groups)
print(f"\nFigure 2 -- ANOVA across models: F={f_stat:.3f}, p={f_p:.6f}")
print(mai_by_model_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3))  # wide + short: avoids excessive height when scaled up in LaTeX
colors = ["tomato" if p_val < 0.05 else "steelblue" for p_val in mai_by_model_df["wilcoxon_p"]]
ax.bar(mai_by_model_df["model"].str.replace("llm_", ""), mai_by_model_df["mean_MAI"],
       color=colors, edgecolor="black")
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_ylabel("Mean MAI")
ax.set_xlabel("Model")
ax.set_title("Marginal MAI by Model")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure2_marginal_mai_by_model.png"), bbox_inches="tight")
plt.close()
print("Saved: figure2_marginal_mai_by_model.png")


# ══════════════════════════════════════════════════════════════════════════
# GLOBAL TUTOR-UTTERANCE CLUSTERING
#
# IMPORTANT: this clustering is fit ONCE on the union of unique tutor
# utterances across ALL FOUR models, then the resulting cluster labels are
# propagated back to every model. This is deliberate, NOT an accident: if
# each model's tutor utterances were clustered independently, "Cluster I"
# in one model's results wouldn't necessarily mean the same thing as
# "Cluster I" in another model's results, making cross-model comparison
# meaningless. Fitting once and sharing labels guarantees the SAME
# real-world groups (e.g. "numeric-dense responses") are being compared
# across all four models.
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}\nGlobal tutor-utterance clustering\n{'=' * 70}")

global_rows, global_embs, seen = [], [], set()
for tag, df in all_dfs.items():
    tut_emb = all_tuts[tag]
    keys = df[["transcriptID", "segment_id"]].drop_duplicates()
    for idx in keys.index:
        key = (df.loc[idx, "transcriptID"], df.loc[idx, "segment_id"])
        if key in seen:
            continue  # this exact (transcript, segment) already added from another model's file
        seen.add(key)
        global_rows.append({
            "transcriptID": key[0], "segment_id": key[1],
            "next_tutor_utterance": df.loc[idx, "next_tutor_utterance"],
        })
        global_embs.append(tut_emb[idx])

global_tutor_df = pd.DataFrame(global_rows)
global_tutor_emb = np.vstack(global_embs)
print(f"Global unique tutor utterances: {len(global_tutor_df)}")

# Standardize before clustering (zero mean, unit variance per dimension) --
# k-means is sensitive to the scale of each embedding dimension otherwise.
scaler = StandardScaler()
global_scaled = scaler.fit_transform(global_tutor_emb)

# Silhouette-score search: try every candidate k, keep whichever one
# produces the most well-separated clusters. Computed on a subsample for
# speed, since silhouette score itself is expensive at full scale.
scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(global_scaled)
    scores.append(silhouette_score(global_scaled, labels, sample_size=2000, random_state=RANDOM_STATE))
global_best_k = list(K_RANGE)[int(np.argmax(scores))]
print(f"Global best k = {global_best_k} (silhouette = {max(scores):.4f})")

km_global = KMeans(n_clusters=global_best_k, random_state=RANDOM_STATE, n_init=10)
global_tutor_df["tutor_cluster_global"] = km_global.fit_predict(global_scaled)

# Propagate the global cluster assignment back onto each model's own
# DataFrame, matched on (transcriptID, segment_id).
cluster_lookup = global_tutor_df[["transcriptID", "segment_id", "tutor_cluster_global"]]
for tag, df in all_dfs.items():
    df.drop(columns=["tutor_cluster"], errors="ignore", inplace=True)
    df_merged = df.merge(cluster_lookup, on=["transcriptID", "segment_id"], how="left")
    df["tutor_cluster"] = df_merged["tutor_cluster_global"].values


# ── Linguistic feature extraction (on the global unique tutor set) ─────────
# These regex-based features are used ONLY to help interpret and LABEL
# what each cluster actually represents (e.g. "the numeric-dense one") --
# they play no role in the clustering itself, which was based purely on
# sentence embeddings.
NUMERIC_WORDS = r"\b(times|plus|minus|divide|divided|multiply|multiplied|equals?|" \
                r"squared|fraction|numerator|denominator|coefficient|exponent|" \
                r"parenthes[ei]s|variable|slope|equation)\b"
PRAISE_WORDS = r"\b(good job|great|awesome|nice|love|perfect|excellent|well done|" \
               r"good work|proud)\b"


def compute_features(text):
    text = str(text)
    return {
        "word_count": len(text.split()),
        "n_numeric_words": len(re.findall(NUMERIC_WORDS, text, flags=re.IGNORECASE)),
        "has_praise": bool(re.search(PRAISE_WORDS, text, flags=re.IGNORECASE)),
        "is_spanish_ish": bool(re.search(r"\b(qu[eé]|c[oó]mo|est[aá]|es|con|para|pero)\b",
                                          text, flags=re.IGNORECASE)),
    }


feats = global_tutor_df["next_tutor_utterance"].apply(compute_features).apply(pd.Series)
global_tutor_df = pd.concat([global_tutor_df.reset_index(drop=True), feats.reset_index(drop=True)], axis=1)

# Identify which raw cluster index (0, 1, 2, ...) corresponds to which
# real-world pattern, so we can label them "I" / "II" / "III" by MEANING
# rather than by an arbitrary number k-means happened to assign.
cluster_profile_id = global_tutor_df.groupby("tutor_cluster_global").agg(
    mean_numeric=("n_numeric_words", "mean"),
    pct_praise=("has_praise", "mean"),
    pct_spanish=("is_spanish_ish", "mean"),
)
cluster_label = {}
cluster_label[cluster_profile_id["mean_numeric"].idxmax()] = "I"    # most numeric-dense cluster
cluster_label[cluster_profile_id["pct_praise"].idxmax()] = "II"     # most praise-dense cluster
cluster_label[cluster_profile_id["pct_spanish"].idxmax()] = "III"   # most Spanish-flagged cluster


# ══════════════════════════════════════════════════════════════════════════
# TABLE: MAI by tutor cluster, WITH COHEN'S D
#
# Cohen's d here is the one-sample standardized effect size: mean MAI
# divided by the SD of MAI within that model x cluster cell. A p-value
# alone can't tell a reader whether "0.007" is a big or small effect --
# Cohen's d answers exactly that question, in units a reader can compare
# against conventional benchmarks (|d| >= 0.2 small, >= 0.5 medium, >= 0.8
# large).
# ══════════════════════════════════════════════════════════════════════════
bar_rows, table_rows = [], []
for tag, df in all_dfs.items():
    df["cluster_label"] = df["tutor_cluster"].map(cluster_label)

    mai_rows = []
    for exp in EXPERIMENTS:
        paired = event_effect_paired_df(df, exp)
        if paired.empty:
            continue
        paired["MAI"] = paired["diff"]
        paired["cluster_label"] = paired["tutor_cluster"].map(cluster_label)
        mai_rows.append(paired)
    master_paired = pd.concat(mai_rows, ignore_index=True)

    for c in ["I", "II", "III"]:
        sub = master_paired[master_paired["cluster_label"] == c]
        if len(sub) == 0:
            continue
        t_stat, p_val = stats.ttest_1samp(sub["MAI"], 0)
        mean_mai, std_mai = sub["MAI"].mean(), sub["MAI"].std()
        cohens_d = mean_mai / std_mai if std_mai > 0 else np.nan  # the effect-size calculation itself

        table_rows.append({
            "model": tag.replace("llm_", ""), "cluster": c, "n": len(sub),
            "mean_MAI": mean_mai, "std_MAI": std_mai,
            "cohens_d": cohens_d, "p_value": p_val,
        })
        # Also stash the mean for the Figure 3 bar chart below.
        bar_rows.append({"model": tag.replace("llm_", ""), "cluster_label": c, "mean_MAI": mean_mai})

table_df = pd.DataFrame(table_rows)
print(f"\n{'=' * 70}\nMAI by tutor cluster, with Cohen's d\n{'=' * 70}")
print(table_df.round(4).to_string(index=False))
table_df.to_csv(os.path.join(OUTPUT_DIR, "table_mai_by_cluster_with_cohens_d.csv"), index=False)


# ══════════════════════════════════════════════════════════════════════════
# TABLE: aggregate similarity + MAI by model x prompting strategy
#
# This is the table underlying the paper's RQ1/RQ2 evidence: raw similarity
# under each condition, plus MAI, event win/gain rate, and a paired
# significance test, one row per model x strategy combination.
# ══════════════════════════════════════════════════════════════════════════
agg_rows = []
for tag in FILE_TAGS:
    for exp in EXPERIMENTS:
        key = (tag, exp)
        if key not in all_paired:
            continue
        paired = all_paired[key]
        event_sim = paired["gen_tutor_sim_event"]
        no_event_sim = paired["gen_tutor_sim_no"]
        diff = paired["diff"]

        w_stat, w_p = stats.wilcoxon(diff)
        gain_rate = (diff > 0).mean()  # proportion of items where "event" beat "no_event"

        agg_rows.append({
            "model": tag.replace("llm_", ""), "experiment": exp,
            "event_mean": event_sim.mean(), "event_sd": event_sim.std(),
            "no_event_mean": no_event_sim.mean(), "no_event_sd": no_event_sim.std(),
            "mai_mean": diff.mean(), "mai_sd": diff.std(),
            "gain_rate": gain_rate, "wilcoxon_p": w_p,
        })

agg_df = pd.DataFrame(agg_rows)
print(f"\n{'=' * 70}\nAggregate similarity + MAI by model x strategy\n{'=' * 70}")
print(agg_df.round(4).to_string(index=False))
agg_df.to_csv(os.path.join(OUTPUT_DIR, "table_aggregate_similarity_and_mai.csv"), index=False)


# ══════════════════════════════════════════════════════════════════════════
# FIGURE 3: MAI by strategy, and MAI by tutor cluster, across models
#
# Two panels side by side in ONE figure (not two separate figures) because
# they're both "how does MAI break down by a second factor, across all
# four models" -- keeping them together makes the comparison between
# "strategy matters" and "response style matters" easy to see at a glance.
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: MAI by prompting strategy, across models
ax = axes[0]
pivot_strategy = agg_df.pivot(index="experiment", columns="model", values="mai_mean").reindex(EXPERIMENTS)
pivot_strategy.plot(kind="bar", ax=ax, legend=True)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_ylabel("Mean MAI")
ax.set_title("(a) MAI by Prompting Strategy")
ax.tick_params(axis="x", rotation=0)

# Right panel: MAI by tutor-response cluster, across models
ax = axes[1]
bar_df = pd.DataFrame(bar_rows)
pivot_cluster = bar_df.pivot(index="cluster_label", columns="model", values="mean_MAI").reindex(["I", "II", "III"])
pivot_cluster.plot(kind="bar", ax=ax, legend=True)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_ylabel("")
ax.set_xlabel("Tutor cluster")
ax.set_title("(b) MAI by Tutor-Response Cluster")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure3_strategy_and_cluster.png"), bbox_inches="tight")
plt.close()
print("\nSaved: figure3_strategy_and_cluster.png")


print(f"\n{'=' * 70}\nAll done. Outputs saved to: {OUTPUT_DIR}\n{'=' * 70}")
print("  figure1_mai_distribution.png       (per-model MAI boxplot)")
print("  figure1_shapiro_results.csv")
print("  figure2_marginal_mai_by_model.png  (marginal MAI, red = significant)")
print("  figure3_strategy_and_cluster.png   (two-panel: strategy | cluster)")
print("  table_aggregate_similarity_and_mai.csv")
print("  table_mai_by_cluster_with_cohens_d.csv")

FileNotFoundError: [Errno 2] No such file or directory: './llm_mistrallatest.csv'